## Practical LLM RL with TRL & GRPO

---

The Post-Training Alignment Landscape (SFT vs. DPO vs. PPO vs. GRPO)

Before writing training pipelines, we need to understand where GRPO fits relative to alternative post-training paradigms, why it was created, and what engineering tradeoffs each method imposes.

---

## 1. The Four Alignment Paradigms

```text
Post-Training Alignment Landscape
```

```text
Offline / Static Data                  Online / Exploration-Driven
─────────────────────               ───────────────────────────────
• SFT (Supervised Fine-Tuning)        • PPO (Proximal Policy Optimization)
• DPO (Direct Preference Opt)        • GRPO (Group Relative Policy Optimization)
```


## The Problem: Learning to Code Without a Solution Manual

Imagine you have a tough programming puzzle:

- Input: a problem description
- Goal: write Python code that passes 5 hidden test cases
- Catch: nobody has written a tutorial or solution for this problem yet; all you have is a test script that outputs PASS or FAIL

Now, compare how DPO and GRPO approach this challenge.


## 2. The Four Alignment Paradigms in Detail

### 1. SFT (Supervised Fine-Tuning) — Behavioral Cloning

#### Engineering mechanism

You curate a dataset of pairs $\mathcal{D} = \{(x, y^*)\}$, where $x$ is the prompt and $y^* = (y^*_1, y^*_2, \dots, y^*_T)$ is the gold-standard token sequence.

You train the network with standard cross-entropy loss to maximize the likelihood of predicting the exact next expert token given the prompt and preceding gold tokens:

$$
\mathcal{L}_{\text{SFT}}(\theta) = -\sum_{t=1}^{T} \log \pi_\theta(y^*_t \mid x, y^*_{<t})
$$

#### Concrete example

- Prompt $(x)$: “Write a function to add two numbers.”
- Target $(y^*)$: “def add(a, b):\n    return a + b”

#### Forward pass

- Feed $[x]$ → predict “def”
- Feed $[x, \text{"def"}]$ → predict “ add”
- Feed $[x, \text{"def"}, \text{" add"}]$ → predict “(a,”

Loss is computed at every position against the target string.

#### Why it works and where it breaks

- Strength: extremely stable and fast to converge
- Best for teaching syntax, output formatting, and domain vocabulary
- Failure mode: exposure bias and covariate shift; at inference time the model uses its own generated tokens, which can lead to compounding errors

### 2. DPO (Direct Preference Optimization) — Static Pairwise Contrast

#### Engineering mechanism

DPO eliminates reinforcement-learning loops by re-parameterizing the reward function directly in terms of the policy.

You provide static triplets $\mathcal{D} = \{(x, y_w, y_l)\}$, where $y_w$ is the winning response and $y_l$ is the losing response.

$$
\mathcal{L}_{\text{DPO}}(\theta) = -\log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right)
$$

#### Concrete example

- Prompt $(x)$: “Is 17 prime?”
- Winning response $(y_w)$: “Yes, 17 has no divisors other than 1 and itself.”
- Losing response $(y_l)$: “No, 17 is divisible by 3.”

#### Forward passes required per step

1. Pass $(x, y_w)$ through the trainable policy to compute $\log \pi_\theta(y_w \mid x)$
2. Pass $(x, y_l)$ through the trainable policy to compute $\log \pi_\theta(y_l \mid x)$
3. Pass $(x, y_w)$ through the frozen reference model to compute $\log \pi_{\text{ref}}(y_w \mid x)$
4. Pass $(x, y_l)$ through the frozen reference model to compute $\log \pi_{\text{ref}}(y_l \mid x)$

DPO increases the probability of $y_w$ relative to the baseline and decreases the probability of $y_l$.

#### Why it works and where it breaks

- Strength: no dynamic rollout generation is needed, so it is lightweight and stable
- Failure mode: no self-exploration; it only re-weights responses already present in the dataset

### 3. PPO (Proximal Policy Optimization) — Full Online Actor-Critic RL

#### Engineering mechanism

PPO is an online, closed-loop RL framework. The policy actively generates completions, a learned reward model scores them, and a critic network $V_\phi$ estimates the expected value of every intermediate state to compute token-level generalized advantage estimation (GAE).

The policy is updated using the clipped surrogate objective:

$$
\mathcal{L}_{\text{PPO}}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) \hat{A}_t, \, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \right) \right]
$$

where the importance sampling ratio is:

$$
r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)}
$$

#### Memory architecture layout

To execute one PPO step on an LLM, you must manage four separate model components in VRAM:

```text
Trainable Models:
1. Actor policy (πθ)
2. Critic network (Vφ)

Frozen Models:
3. Reference model (πref)
4. Reward model (Rψ)
```

#### Why it works and where it breaks

- Strength: handles multi-turn agent environments and complex reward landscapes
- Failure mode: high VRAM footprint and instability if the critic provides noisy baselines

### 4. GRPO (Group Relative Policy Optimization) — Critic-Less Group Online RL

#### Engineering mechanism

GRPO keeps the online exploration benefits of RL but discards the critic network entirely.

For each prompt $q$, sample a group of $G$ candidate outputs: $\{o_1, o_2, \dots, o_G\}$.

Score each output with a deterministic verifier or reward function: $\{r_1, r_2, \dots, r_G\}$.

Compute the baseline directly from the sample statistics of the group:

$$
A_i = \frac{r_i - \text{mean}(\{r_1, \dots, r_G\})}{\text{std}(\{r_1, \dots, r_G\}) + \epsilon}
$$

Optimize the policy with the clipped surrogate loss and an explicit token-level KL divergence penalty against the reference model $\pi_{\text{ref}}$.

#### Concrete execution example

- Prompt $(q)$: “Find $x$ if $3x + 9 = 24$”
- Sample $G = 4$ parallel rollouts from the actor

```text
Rollout 1: 3x = 15 -> x = 5            -> Reward r1 = 1.0
Rollout 2: 3x = 33 -> x = 11           -> Reward r2 = 0.0
Rollout 3: 3x = 15 -> x = 5            -> Reward r3 = 1.0
Rollout 4: x = 24 - 9 = 15             -> Reward r4 = 0.0
```

Group stats:

- Mean = 0.5
- Std = 0.5
- Advantages for rollouts 1 and 3: $+1.0$
- Advantages for rollouts 2 and 4: $-1.0$

#### Why it works and where it breaks

- Strength: eliminates much of the training memory footprint by dropping the critic
- Failure mode: if the group yields identical rewards, the advantage becomes zero and no useful gradient update occurs

### Comparative architecture reference

| Dimension | SFT | DPO | PPO | GRPO |
| --- | --- | --- | --- | --- |
| Optimization type | Maximum likelihood (supervised) | Implicit reward contrast (supervised) | Online RL (policy gradient) | Online RL (policy gradient) |
| Data generation | Static targets | Static pairs | Dynamic live rollouts | Dynamic live rollouts |
| Requires critic network? | No | No | Yes (high VRAM) | No (group stats) |
| Baseline computation | None | Implicit via $\pi_{\text{ref}}$ | Value model $V_\phi(s_t)$ | Normalized group mean |
| Exploration capability | None | None | High | High |
| Primary system bottleneck | Quality of curated data | Pairwise data coverage | VRAM and critic stability | Group diversity and reward quality |
